In [3]:
import pandas as pd
import numpy as np
from pybaseball import batting_stats
import warnings
warnings.filterwarnings('ignore')

# Pull batting stats for all seasons 2004-2024
all_seasons = []

for year in range(2005, 2026):
    print(f"Fetching {year}...")
    try:
        df = batting_stats(year, qual=50)  # min 50 PA to filter out garbage samples
        df['season'] = year
        all_seasons.append(df)
    except Exception as e:
        print(f"Error fetching {year}: {e}")

raw_data = pd.concat(all_seasons, ignore_index=True)
print(f"\nTotal records: {len(raw_data)}")
print(f"Seasons: {raw_data['season'].min()} - {raw_data['season'].max()}")
print(f"Columns: {len(raw_data.columns)}")
raw_data.head()

Fetching 2005...
Fetching 2006...
Fetching 2007...
Fetching 2008...
Fetching 2009...
Fetching 2010...
Fetching 2011...
Fetching 2012...
Fetching 2013...
Fetching 2014...
Fetching 2015...
Fetching 2016...
Fetching 2017...
Fetching 2018...
Fetching 2019...
Fetching 2020...
Fetching 2021...
Fetching 2022...
Fetching 2023...
Fetching 2024...
Fetching 2025...

Total records: 11403
Seasons: 2005 - 2025
Columns: 321


,IDfg,Season,Name,Team,Age,G,AB,PA,H,1B,...,HardHit,HardHit%,Events,CStr%,CSW%,xBA,xSLG,xwOBA,L-WAR,season
0,1274,2005,Alex Rodriguez,NYY,29,162,605,715,194,116,...,NaN,NaN,0,0.162,0.272,NaN,NaN,NaN,9.1,2005
1,96,2005,Andruw Jones,ATL,28,160,586,672,154,76,...,NaN,NaN,0,0.134,0.268,NaN,NaN,NaN,7.9,2005
2,1177,2005,Albert Pujols,STL,25,161,591,700,195,114,...,NaN,NaN,0,0.169,0.228,NaN,NaN,NaN,7.7,2005
3,1679,2005,Chase Utley,PHI,26,147,543,628,158,85,...,NaN,NaN,0,0.205,0.279,NaN,NaN,NaN,7.2,2005
4,525,2005,Derrek Lee,CHC,29,158,594,691,199,100,...,NaN,NaN,0,0.158,0.245,NaN,NaN,NaN,7.0,2005


In [4]:
# Save raw data first
raw_data.to_csv('raw_batting_data.csv', index=False)
print(f"Raw data saved. Shape: {raw_data.shape}")
print(f"\nAvailable columns sample:")
print(list(raw_data.columns[:50]))

Raw data saved. Shape: (11403, 321)

Available columns sample:
['IDfg', 'Season', 'Name', 'Team', 'Age', 'G', 'AB', 'PA', 'H', '1B', '2B', '3B', 'HR', 'R', 'RBI', 'BB', 'IBB', 'SO', 'HBP', 'SF', 'SH', 'GDP', 'SB', 'CS', 'AVG', 'GB', 'FB', 'LD', 'IFFB', 'Pitches', 'Balls', 'Strikes', 'IFH', 'BU', 'BUH', 'BB%', 'K%', 'BB/K', 'OBP', 'SLG', 'OPS', 'ISO', 'BABIP', 'GB/FB', 'LD%', 'GB%', 'FB%', 'IFFB%', 'HR/FB', 'IFH%']


In [5]:
# Feature Engineering
def engineer_features(df):
    df = df.copy()
    
    # --- Plate discipline ratios ---
    df['BB_rate'] = df['BB'] / df['PA']
    df['K_rate'] = df['SO'] / df['PA']
    df['BB_K_ratio'] = df['BB'] / (df['SO'] + 1)
    df['HBP_rate'] = df['HBP'] / df['PA']
    df['IBB_rate'] = df['IBB'] / df['PA']
    
    # --- Power metrics ---
    df['ISO'] = df['SLG'] - df['AVG']
    df['HR_rate'] = df['HR'] / df['PA']
    df['XBH_rate'] = (df['2B'] + df['3B'] + df['HR']) / df['PA']
    df['HR_FB_proxy'] = df['HR'] / (df['AB'] - df['SO'] + 1)
    df['SLG_minus_OBP'] = df['SLG'] - df['OBP']
    
    # --- Contact metrics ---
    df['BABIP_proxy'] = (df['H'] - df['HR']) / (df['AB'] - df['SO'] - df['HR'] + df['SF'] + 1)
    df['contact_rate'] = 1 - df['K_rate']
    df['singles_rate'] = df['1B'] / df['PA']
    df['doubles_rate'] = df['2B'] / df['PA']
    df['triples_rate'] = df['3B'] / df['PA']
    
    # --- Speed/baserunning ---
    df['SB_rate'] = df['SB'] / df['G']
    df['SB_success'] = df['SB'] / (df['SB'] + df['CS'] + 1)
    
    # --- Volume/durability ---
    df['AB_per_game'] = df['AB'] / df['G']
    df['PA_per_game'] = df['PA'] / df['G']
    
    # --- Efficiency ratios ---
    df['OPS'] = df['OBP'] + df['SLG']
    df['OPS_plus_proxy'] = df['OPS'] / df['OPS'].mean()
    df['RBI_per_PA'] = df['RBI'] / df['PA']
    df['R_per_PA'] = df['R'] / df['PA']
    
    # --- Age curves ---
    df['age_squared'] = df['Age'] ** 2
    df['age_prime'] = ((df['Age'] - 27) ** 2) * -1
    
    # --- Statcast metrics (post-2015, NaN for earlier) ---
    statcast_cols = ['HardHit%', 'xBA', 'xSLG', 'xwOBA', 'EV', 'LA', 'Barrel%']
    for col in statcast_cols:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].median())
    
    return df

featured_data = engineer_features(raw_data)
print(f"Features after engineering: {len(featured_data.columns)}")
featured_data.head()

Features after engineering: 344


,IDfg,Season,Name,Team,Age,G,AB,PA,H,1B,...,triples_rate,SB_rate,SB_success,AB_per_game,PA_per_game,OPS_plus_proxy,RBI_per_PA,R_per_PA,age_squared,age_prime
0,1274,2005,Alex Rodriguez,NYY,29,162,605,715,194,116,...,0.001399,0.129630,0.750000,3.734568,4.413580,1.504723,0.181818,0.173427,841,-4
1,96,2005,Andruw Jones,ATL,28,160,586,672,154,76,...,0.004464,0.031250,0.555556,3.662500,4.200000,1.345640,0.190476,0.141369,784,-1
2,1177,2005,Albert Pujols,STL,25,161,591,700,195,114,...,0.002857,0.099379,0.842105,3.670807,4.347826,1.516399,0.167143,0.184286,625,-4
3,1679,2005,Chase Utley,PHI,26,147,543,628,158,85,...,0.009554,0.108844,0.800000,3.693878,4.272109,1.336883,0.167197,0.148089,676,-1
4,525,2005,Derrek Lee,CHC,29,158,594,691,199,100,...,0.004342,0.094937,0.789474,3.759494,4.373418,1.576238,0.154848,0.173661,841,-4


In [6]:
# Create target variable — next season WAR
def create_target(df):
    df = df.copy()
    
    # Sort by player and season
    df = df.sort_values(['Name', 'season'])
    
    # Shift WAR forward by 1 season per player
    df['next_WAR'] = df.groupby('Name')['WAR'].shift(-1)
    df['next_season'] = df.groupby('Name')['season'].shift(-1)
    
    # Only keep rows where next season is exactly 1 year later
    # This prevents gaps (e.g. player misses a season) from being treated as valid pairs
    df = df[df['next_season'] - df['season'] == 1]
    
    # Drop rows with missing target
    df = df.dropna(subset=['next_WAR'])
    
    return df

model_data = create_target(featured_data)
print(f"Rows after target alignment: {len(model_data)}")
print(f"Season range: {model_data['season'].min()} - {model_data['season'].max()}")
print(f"\nTarget variable stats:")
print(model_data['next_WAR'].describe())

Rows after target alignment: 8352
Season range: 2005 - 2024

Target variable stats:
count    8352.000000
mean        1.242205
std         1.848612
min        -3.100000
25%        -0.100000
50%         0.700000
75%         2.200000
max        11.300000
Name: next_WAR, dtype: float64


In [8]:
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.feature_selection import SequentialFeatureSelector
import numpy as np

# Define feature columns — exclude non-numeric and leakage columns
exclude_cols = ['IDfg', 'Season', 'Name', 'Team', 'season', 'next_WAR', 
                'next_season', 'WAR', 'L-WAR']

feature_cols = [col for col in model_data.columns 
                if col not in exclude_cols 
                and model_data[col].dtype in ['float64', 'int64']]

print(f"Features going into model: {len(feature_cols)}")

# Build X and y
X = model_data[feature_cols].copy()
y = model_data['next_WAR'].copy()

# Fill any remaining NaNs with column median
X = X.fillna(X.median())
X = X.fillna(0)  # catch any columns where median is also NaN
print(f"Any NaNs remaining: {X.isna().sum().sum()}")

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=feature_cols)

print(f"X shape: {X_scaled.shape}")
print(f"y shape: {y.shape}")
print("\nRunning Sequential Forward Feature Selection...")
print("This will take 2-3 minutes...")

# Ridge with cross-validation to find best alpha
ridge = RidgeCV(alphas=[0.1, 1.0, 10.0, 100.0], cv=5)

# Sequential forward selection — find top 20 features
sfs = SequentialFeatureSelector(
    ridge, 
    n_features_to_select=20,
    direction='forward',
    scoring='neg_mean_squared_error',
    cv=5
)

sfs.fit(X_scaled, y)

selected_features = X_scaled.columns[sfs.get_support()].tolist()
print(f"\nTop 20 selected features:")
for i, feat in enumerate(selected_features, 1):
    print(f"  {i}. {feat}")

Features going into model: 335
Any NaNs remaining: 0
X shape: (8352, 335)
y shape: (8352,)

Running Sequential Forward Feature Selection...
This will take 2-3 minutes...

Top 20 selected features:
  1. Age
  2. IBB
  3. BU
  4. BABIP
  5. Pos
  6. RAR
  7. Contact%
  8. UBR
  9. Med%
  10. Zone% (pi)
  11. FRM
  12. Oppo%+
  13. Hard%+
  14. maxEV
  15. IBB_rate
  16. HR_FB_proxy
  17. BABIP_proxy
  18. SB_rate
  19. AB_per_game
  20. PA_per_game


In [9]:
# Rolling backtesting framework
results = []

seasons = sorted(model_data['season'].unique())
# Need at least 3 seasons of training data before predicting
test_seasons = seasons[3:]

for test_season in test_seasons:
    # Training data = all seasons before test season
    train_mask = model_data['season'] < test_season
    test_mask = model_data['season'] == test_season
    
    X_train = X_scaled[train_mask.values][selected_features]
    y_train = y[train_mask.values]
    X_test = X_scaled[test_mask.values][selected_features]
    y_test = y[test_mask.values]
    
    if len(X_test) == 0:
        continue
    
    # Fit model on training data only
    ridge_final = RidgeCV(alphas=[0.1, 1.0, 10.0, 100.0], cv=5)
    ridge_final.fit(X_train, y_train)
    
    # Predict on test season
    preds = ridge_final.predict(X_test)
    mse = mean_squared_error(y_test, preds)
    
    results.append({
        'test_season': test_season,
        'mse': mse,
        'n_test': len(X_test),
        'mean_actual': y_test.mean(),
        'mean_predicted': preds.mean()
    })
    
    print(f"Season {test_season}: MSE={mse:.2f}, n={len(X_test)}")

results_df = pd.DataFrame(results)
avg_mse = results_df['mse'].mean()
print(f"\nAverage MSE across all seasons: {avg_mse:.2f}")
print(f"Average RMSE: {np.sqrt(avg_mse):.2f} WAR")

Season 2008: MSE=2.36, n=434
Season 2009: MSE=2.35, n=431
Season 2010: MSE=2.64, n=434
Season 2011: MSE=2.26, n=440
Season 2012: MSE=2.17, n=439
Season 2013: MSE=2.18, n=434
Season 2014: MSE=2.39, n=424
Season 2015: MSE=2.01, n=416
Season 2016: MSE=2.21, n=406
Season 2017: MSE=1.93, n=409
Season 2018: MSE=2.06, n=429
Season 2019: MSE=1.67, n=367
Season 2020: MSE=3.03, n=359
Season 2021: MSE=2.16, n=414
Season 2022: MSE=1.82, n=423
Season 2023: MSE=2.23, n=420
Season 2024: MSE=1.87, n=413

Average MSE across all seasons: 2.20
Average RMSE: 1.48 WAR


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Final model on all data for outlier analysis
X_final = X_scaled[selected_features]
ridge_full = RidgeCV(alphas=[0.1, 1.0, 10.0, 100.0], cv=5)
ridge_full.fit(X_final, y)
all_preds = ridge_full.predict(X_final)

# Build results dataframe
analysis_df = model_data[['Name', 'season', 'next_WAR']].copy()
analysis_df['predicted_WAR'] = all_preds
analysis_df['error'] = analysis_df['next_WAR'] - analysis_df['predicted_WAR']
analysis_df['abs_error'] = analysis_df['error'].abs()

# Plot 1 — MSE by season
plt.figure(figsize=(12, 4))
plt.plot(results_df['test_season'], results_df['mse'], marker='o', color='steelblue')
plt.axhline(avg_mse, color='red', linestyle='--', label=f'Avg MSE: {avg_mse:.2f}')
plt.title('MSE by Season — Rolling Backtest')
plt.xlabel('Test Season')
plt.ylabel('MSE')
plt.legend()
plt.tight_layout()
plt.savefig('mse_by_season.png', dpi=150)
plt.show()

# Plot 2 — Actual vs Predicted
plt.figure(figsize=(8, 6))
plt.scatter(analysis_df['next_WAR'], analysis_df['predicted_WAR'], 
            alpha=0.3, color='steelblue')
plt.plot([-3, 12], [-3, 12], 'r--', label='Perfect prediction')
plt.xlabel('Actual WAR')
plt.ylabel('Predicted WAR')
plt.title('Actual vs Predicted Next-Season WAR')
plt.legend()
plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=150)
plt.show()

# Top over and underperformers
print("Biggest overperformers (did much better than predicted):")
print(analysis_df.nlargest(10, 'error')[['Name', 'season', 'next_WAR', 'predicted_WAR', 'error']].to_string())

print("\nBiggest underperformers (did much worse than predicted):")
print(analysis_df.nsmallest(10, 'error')[['Name', 'season', 'next_WAR', 'predicted_WAR', 'error']].to_string())